# Esports Latency Trader Screen

Visual companion to `notes/esports_latency_trader_screen.md`.

This notebook reads the cached DuckDB/parquet screen artifacts from `data/analysis/esports_latency_traders/` and renders the main trader visuals: candidate universe, aggregate score leaderboard, stale pre-snap fingerprints, PnL/volume scatter, and market-type concentration.

The key caveat carries through every chart: this uses a **price-snap proxy**, not true in-game event timestamps. A trader buying before the first $0.90 print is suspicious only after validating the actual match event time and maker/taker role.

In [ ]:

from pathlib import Path
from html import escape
import math

import duckdb
import pandas as pd
from IPython.display import HTML, display

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 180)

# Resolve research directory whether executed from repo root or the notebooks folder.
candidates = []
for p in [Path.cwd(), *Path.cwd().parents]:
    candidates.append(p)
    candidates.append(p / "polymarket" / "research")
    candidates.append(p / "research")

RESEARCH_DIR = None
for p in candidates:
    if (p / "data").exists() and (p / "notes").exists() and p.name == "research":
        RESEARCH_DIR = p.resolve()
        break

if RESEARCH_DIR is None:
    raise RuntimeError("Could not locate polymarket/research directory from current working directory")

CACHE_DIR = RESEARCH_DIR / "data" / "analysis" / "esports_latency_traders"
required = [
    "candidate_market_universe.parquet",
    "candidate_market_tokens.parquet",
    "trader_position_metrics.parquet",
    "trader_latency_fingerprint.parquet",
    "ranked_trader_candidates.parquet",
    "ranked_primary_traders_aggregate.parquet",
]
missing = [name for name in required if not (CACHE_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f"Missing cached screen artifacts in {CACHE_DIR}: {missing}")

con = duckdb.connect()
print(f"Research dir: {RESEARCH_DIR}")
print(f"Cache dir:    {CACHE_DIR}")


In [ ]:

def q(sql: str) -> pd.DataFrame:
    return con.execute(sql).fetchdf()

universe = q(f"SELECT * FROM read_parquet('{CACHE_DIR / 'candidate_market_universe.parquet'}')")
ranked = q(f"SELECT * FROM read_parquet('{CACHE_DIR / 'ranked_trader_candidates.parquet'}')")
agg = q(f"SELECT * FROM read_parquet('{CACHE_DIR / 'ranked_primary_traders_aggregate.parquet'}')")

# Clean up infinities for display.
for df in [ranked, agg]:
    for col in df.select_dtypes(include="number").columns:
        df[col] = df[col].replace([float("inf"), float("-inf")], pd.NA)

summary = pd.DataFrame({
    "dataset": [
        "candidate markets",
        "ranked trader cells",
        "primary prop candidate addresses",
    ],
    "rows": [len(universe), len(ranked), len(agg)],
    "distinct_addresses": [pd.NA, ranked["address"].nunique(), agg["address"].nunique()],
})
summary


In [ ]:

CSS = '\n<style>\n.pm-wrap { font-family: -apple-system, BlinkMacSystemFont, Segoe UI, sans-serif; color: #17202a; }\n.pm-card { border: 1px solid #d9e2ec; border-radius: 6px; padding: 14px 16px; margin: 12px 0; background: #fbfdff; }\n.pm-title { font-weight: 700; font-size: 16px; margin-bottom: 4px; }\n.pm-subtitle { color: #52616f; font-size: 12px; margin-bottom: 10px; }\n.pm-table { border-collapse: collapse; font-size: 12px; width: 100%; }\n.pm-table th { text-align: left; background: #eef3f8; color: #243b53; border-bottom: 1px solid #cbd8e3; padding: 6px 8px; }\n.pm-table td { border-bottom: 1px solid #e7edf3; padding: 5px 8px; white-space: nowrap; }\n.pm-note { color: #52616f; font-size: 12px; margin-top: 8px; }\n</style>\n'

def money(x):
    if pd.isna(x):
        return ""
    return f"${x:,.0f}"

def num(x, digits=3):
    if pd.isna(x):
        return ""
    if isinstance(x, (int, float)) and abs(x) >= 1000:
        return f"{x:,.0f}"
    if isinstance(x, float):
        return f"{x:.{digits}f}".rstrip("0").rstrip(".")
    return str(x)

def short_addr(addr, n=10):
    if not isinstance(addr, str):
        return ""
    return addr[:n] + "..." + addr[-4:]

def html_table(df, columns=None, formatters=None, max_rows=20):
    columns = columns or list(df.columns)
    formatters = formatters or {}
    rows = []
    for _, r in df.head(max_rows).iterrows():
        tds = []
        for c in columns:
            v = r[c]
            if c in formatters:
                v = formatters[c](v)
            elif isinstance(v, float):
                v = num(v)
            elif pd.isna(v):
                v = ""
            tds.append(f"<td>{escape(str(v))}</td>")
        rows.append("<tr>" + "".join(tds) + "</tr>")
    head = "".join(f"<th>{escape(str(c))}</th>" for c in columns)
    return CSS + f"<table class='pm-table'><thead><tr>{head}</tr></thead><tbody>{''.join(rows)}</tbody></table>"

def svg_bar(df, label_col, value_col, title, subtitle="", width=980, row_h=24, value_fmt=None, color="#2f80ed"):
    data = df[[label_col, value_col]].copy().dropna()
    data[value_col] = pd.to_numeric(data[value_col], errors="coerce").fillna(0)
    data = data[data[value_col] >= 0]
    max_v = max(float(data[value_col].max()), 1.0) if len(data) else 1.0
    left, right = 235, 110
    top = 46
    height = top + row_h * len(data) + 20
    parts = [CSS, f"<div class='pm-card'><div class='pm-title'>{escape(title)}</div><div class='pm-subtitle'>{escape(subtitle)}</div>"]
    parts.append(f"<svg width='{width}' height='{height}' viewBox='0 0 {width} {height}' role='img'>")
    for i, (_, r) in enumerate(data.iterrows()):
        y = top + i * row_h
        label = str(r[label_col])
        val = float(r[value_col])
        bar_w = (width - left - right) * val / max_v
        parts.append(f"<text x='10' y='{y+15}' font-size='12' fill='#243b53'>{escape(label[:42])}</text>")
        parts.append(f"<rect x='{left}' y='{y+4}' width='{bar_w:.1f}' height='14' rx='2' fill='{color}' opacity='0.82'></rect>")
        shown = value_fmt(val) if value_fmt else num(val)
        parts.append(f"<text x='{left+bar_w+6}' y='{y+15}' font-size='12' fill='#243b53'>{escape(shown)}</text>")
    parts.append("</svg></div>")
    return "".join(parts)

def svg_scatter(df, x_col, y_col, size_col, color_col, title, subtitle="", label_col=None, width=980, height=560):
    data = df[[x_col, y_col, size_col, color_col] + ([label_col] if label_col else [])].copy().dropna(subset=[x_col, y_col])
    if data.empty:
        return CSS + "<div class='pm-card'>No data</div>"
    for c in [x_col, y_col, size_col, color_col]:
        data[c] = pd.to_numeric(data[c], errors="coerce")
    data = data.dropna(subset=[x_col, y_col])
    x_min, x_max = float(data[x_col].min()), float(data[x_col].max())
    y_min, y_max = float(data[y_col].min()), float(data[y_col].max())
    if x_min == x_max:
        x_min -= 0.01; x_max += 0.01
    if y_min == y_max:
        y_min -= 0.01; y_max += 0.01
    x_pad = (x_max - x_min) * 0.06
    y_pad = (y_max - y_min) * 0.08
    x_min -= x_pad; x_max += x_pad
    y_min = max(0, y_min - y_pad); y_max += y_pad
    left, right, top, bottom = 76, 28, 42, 58
    plot_w, plot_h = width - left - right, height - top - bottom
    size_max = max(float(data[size_col].max()), 1.0)
    color_min = float(data[color_col].min()) if data[color_col].notna().any() else 0.0
    color_max = float(data[color_col].max()) if data[color_col].notna().any() else 1.0
    def sx(x): return left + (float(x) - x_min) / (x_max - x_min) * plot_w
    def sy(y): return top + plot_h - (float(y) - y_min) / (y_max - y_min) * plot_h
    def radius(v): return 4 + 15 * math.sqrt(max(float(v), 0) / size_max)
    def color(v):
        if pd.isna(v) or color_max == color_min:
            return "#2f80ed"
        t = (float(v) - color_min) / (color_max - color_min)
        if t < 0.5:
            u = t / 0.5
            r = int(47 + (39 - 47) * u); g = int(128 + (174 - 128) * u); b = int(237 + (96 - 237) * u)
        else:
            u = (t - 0.5) / 0.5
            r = int(39 + (242 - 39) * u); g = int(174 + (153 - 174) * u); b = int(96 + (74 - 96) * u)
        return f"#{r:02x}{g:02x}{b:02x}"
    parts = [CSS, f"<div class='pm-card'><div class='pm-title'>{escape(title)}</div><div class='pm-subtitle'>{escape(subtitle)}</div>"]
    parts.append(f"<svg width='{width}' height='{height}' viewBox='0 0 {width} {height}' role='img'>")
    for i in range(6):
        x = left + plot_w * i / 5
        val = x_min + (x_max - x_min) * i / 5
        parts.append(f"<line x1='{x}' y1='{top}' x2='{x}' y2='{top+plot_h}' stroke='#e7edf3'></line>")
        parts.append(f"<text x='{x}' y='{top+plot_h+22}' font-size='11' text-anchor='middle' fill='#52616f'>{val:.2f}</text>")
    for i in range(6):
        y = top + plot_h * i / 5
        val = y_max - (y_max - y_min) * i / 5
        parts.append(f"<line x1='{left}' y1='{y}' x2='{left+plot_w}' y2='{y}' stroke='#e7edf3'></line>")
        parts.append(f"<text x='{left-10}' y='{y+4}' font-size='11' text-anchor='end' fill='#52616f'>{val:.2f}</text>")
    parts.append(f"<line x1='{left}' y1='{top+plot_h}' x2='{left+plot_w}' y2='{top+plot_h}' stroke='#7b8794'></line>")
    parts.append(f"<line x1='{left}' y1='{top}' x2='{left}' y2='{top+plot_h}' stroke='#7b8794'></line>")
    parts.append(f"<text x='{left + plot_w/2}' y='{height-18}' font-size='12' text-anchor='middle' fill='#243b53'>{escape(x_col)}</text>")
    parts.append(f"<text transform='translate(18 {top + plot_h/2}) rotate(-90)' font-size='12' text-anchor='middle' fill='#243b53'>{escape(y_col)}</text>")
    for _, r in data.iterrows():
        x, y = sx(r[x_col]), sy(r[y_col])
        rad = radius(r[size_col])
        c = color(r[color_col])
        tip = escape(f"{label_col + ': ' if label_col else ''}{r[label_col] if label_col else ''} {x_col}={r[x_col]:.3f}, {y_col}={r[y_col]:.3f}, {size_col}={r[size_col]:,.0f}, {color_col}={r[color_col]:.3f}")
        parts.append(f"<circle cx='{x:.1f}' cy='{y:.1f}' r='{rad:.1f}' fill='{c}' opacity='0.68' stroke='#102a43' stroke-opacity='0.25'><title>{tip}</title></circle>")
    if label_col:
        label_data = data.sort_values(color_col, ascending=False).head(8)
        for _, r in label_data.iterrows():
            parts.append(f"<text x='{sx(r[x_col])+7:.1f}' y='{sy(r[y_col])-7:.1f}' font-size='10' fill='#102a43'>{escape(str(r[label_col])[:12])}</text>")
    parts.append("</svg></div>")
    return "".join(parts)


## Candidate Market Surface

The universe keeps primary LoL/Dota intragame props separate from controls. The primary set is where threshold/objective resolution gives a plausible latency window; controls help reveal ordinary end-of-market repricing patterns.

In [ ]:

universe_summary = (
    universe.groupby(["candidate_bucket", "game", "market_type_proxy"], dropna=False)
    .agg(markets=("condition_id", "count"), closed_markets=("closed", "sum"), gamma_volume=("volume", "sum"))
    .reset_index()
    .sort_values(["candidate_bucket", "gamma_volume"], ascending=[True, False])
)

bar_data = universe_summary.assign(
    label=lambda d: d["candidate_bucket"].str.replace("_", " ") + " | " + d["game"] + " | " + d["market_type_proxy"].str.replace("_", " ")
).sort_values("gamma_volume", ascending=False).head(14)

display(HTML(svg_bar(bar_data, "label", "gamma_volume", "Candidate Universe by Gamma Volume", "Market classes included in the screen", value_fmt=money, color="#2f80ed")))
display(HTML(html_table(universe_summary, max_rows=20, formatters={"gamma_volume": money})))


## Aggregate Primary Prop Leaderboard

These are address-level aggregates across only LoL/Dota first blood, live total kills, objectives, and multikill props. High score combines realised profitability, repeated coverage, and price-snap fingerprints.

In [ ]:

agg_view = agg.sort_values("aggregate_latency_score", ascending=False).copy()
agg_view["short_address"] = agg_view["address"].map(short_addr)
agg_view["label"] = agg_view["short_address"]

leader_bar = agg_view.head(20)[["label", "aggregate_latency_score"]]
display(HTML(svg_bar(leader_bar, "label", "aggregate_latency_score", "Top Aggregate Primary Prop Scores", "Address-level ranker across LoL/Dota latency-prop buckets", value_fmt=lambda x: f"{x:.3f}", color="#27ae60")))

cols = ["short_address", "n_cells", "games", "n_positions", "gross_usd_volume", "realised_pnl", "win_rate", "profit_factor", "near300_capture_rate", "pre_snap_capture_rate", "stale_capture_rate", "avg_edge_to_90_cents", "aggregate_latency_score"]
formatters = {"gross_usd_volume": money, "realised_pnl": money}
display(HTML(html_table(agg_view[cols], formatters=formatters, max_rows=15)))


## Win Rate vs Stale Pre-Snap Capture

This scatter is the fastest way to separate two stories:

- upper-left / high-score names with huge win rate but low stale-entry share are profitable automated specialists;
- higher stale-capture names are better event-latency validation targets, even when sample size is smaller.

In [ ]:

scatter_df = agg_view.copy()
scatter_df["stale_capture_pct"] = scatter_df["stale_capture_rate"].fillna(0)
scatter_df["gross_usd"] = scatter_df["gross_usd_volume"].fillna(0)

display(HTML(svg_scatter(
    scatter_df,
    x_col="win_rate",
    y_col="stale_capture_pct",
    size_col="gross_usd",
    color_col="aggregate_latency_score",
    label_col="short_address",
    title="Win Rate vs Stale Pre-Snap Capture",
    subtitle="Bubble size = gross USD; color = aggregate score. Tooltip contains exact values.",
)))


## Stale Entry Subset

A stricter view: at least five buys of the eventual winning token before the first $0.90 print, with entry price at $0.80 or lower. This is the subset to hand-label against match timelines first.

In [ ]:

stale = agg_view[agg_view["stale_pre_snap_winner_buys"].fillna(0) >= 5].copy()
stale["label"] = stale["short_address"]
stale = stale.sort_values(["stale_pre_snap_winner_buys", "aggregate_latency_score"], ascending=[False, False])

display(HTML(svg_bar(
    stale.head(20),
    "label",
    "stale_pre_snap_winner_buys",
    "Stale Pre-Snap Winner Buys",
    "Counts of winning-token buys before $0.90 where entry <= $0.80",
    value_fmt=lambda x: f"{x:,.0f}",
    color="#f2994a",
)))

stale_cols = ["short_address", "games", "n_positions", "gross_usd_volume", "realised_pnl", "win_rate", "profit_factor", "stale_pre_snap_winner_buys", "pre_snap_capture_rate", "avg_pre_snap_entry_price", "avg_edge_to_90_cents", "aggregate_latency_score"]
display(HTML(html_table(stale[stale_cols].sort_values("aggregate_latency_score", ascending=False), formatters=formatters, max_rows=20)))


## PnL vs Volume

This chart helps avoid over-weighting tiny perfect records and also surfaces large profitable accounts whose stale-entry ratio is more modest.

In [ ]:

pnl_df = agg_view.copy()
pnl_df["positive_pnl"] = pnl_df["realised_pnl"].clip(lower=0)

display(HTML(svg_scatter(
    pnl_df,
    x_col="gross_usd_volume",
    y_col="realised_pnl",
    size_col="n_positions",
    color_col="stale_capture_rate",
    label_col="short_address",
    title="Primary Prop PnL vs Gross USD",
    subtitle="Bubble size = position count; color = stale capture rate.",
)))


## Cell-Level Concentration

The cell-level view shows which game and market type creates the signal. This matters because a wallet can look great in aggregate while the actual fingerprint comes from one narrow prop family.

In [ ]:

primary_cells = ranked[ranked["candidate_bucket"] == "primary_latency_prop"].copy()
primary_cells["short_address"] = primary_cells["address"].map(short_addr)

cell_summary = (
    primary_cells.groupby(["game", "market_type_proxy"], dropna=False)
    .agg(
        ranked_cells=("address", "count"),
        addresses=("address", "nunique"),
        positions=("n_positions", "sum"),
        gross_usd=("gross_usd_volume", "sum"),
        pnl=("realised_pnl", "sum"),
        avg_win_rate=("win_rate", "mean"),
        avg_score=("latency_suspicion_score", "mean"),
        stale_buys=("stale_pre_snap_winner_buys", "sum"),
    )
    .reset_index()
    .sort_values("gross_usd", ascending=False)
)

cell_bar = cell_summary.assign(label=lambda d: d["game"] + " | " + d["market_type_proxy"].str.replace("_", " "))
display(HTML(svg_bar(cell_bar, "label", "gross_usd", "Primary Cell Gross USD by Game/Market Type", "Where ranked prop specialists concentrated their volume", value_fmt=money, color="#7e57c2")))

top_cells = primary_cells.sort_values("latency_suspicion_score", ascending=False).head(25)
cell_cols = ["short_address", "game", "market_type_proxy", "n_positions", "n_markets", "gross_usd_volume", "realised_pnl", "win_rate", "profit_factor", "winner_buy_actions", "pre_snap_winner_buys", "stale_pre_snap_winner_buys", "near300_snap_winner_buys", "pre_snap_capture_rate", "near300_capture_rate", "avg_edge_to_90_cents", "latency_suspicion_score"]
display(HTML(html_table(top_cells[cell_cols], formatters=formatters, max_rows=25)))


## Control Buckets

Controls prove the caution point: high win rate and price-snap timing also appear in terminal winner markets and CS2 moneyline markets. That does not make them irrelevant, but it makes them less clean as latency-arb evidence.

In [ ]:

controls = ranked[ranked["candidate_bucket"] != "primary_latency_prop"].copy()
controls["short_address"] = controls["address"].map(short_addr)
controls = controls.sort_values("latency_suspicion_score", ascending=False)
control_cols = ["candidate_bucket", "short_address", "game", "market_type_proxy", "n_positions", "n_markets", "gross_usd_volume", "realised_pnl", "win_rate", "profit_factor", "near300_capture_rate", "avg_edge_to_90_cents", "latency_suspicion_score"]
display(HTML(html_table(controls[control_cols], formatters=formatters, max_rows=20)))


## Drilldown Helper

Set `TARGET_ADDRESS` to any full wallet address from the report to inspect its ranked cells. The default uses the highest aggregate score.

In [ ]:

TARGET_ADDRESS = agg_view.iloc[0]["address"]

drill = ranked[ranked["address"].str.lower() == TARGET_ADDRESS.lower()].copy().sort_values("latency_suspicion_score", ascending=False)
print(TARGET_ADDRESS)
display(HTML(html_table(drill[["candidate_bucket", "game", "market_type_proxy", "n_positions", "n_markets", "gross_usd_volume", "realised_pnl", "win_rate", "profit_factor", "winner_buy_actions", "pre_snap_winner_buys", "stale_pre_snap_winner_buys", "near300_snap_winner_buys", "avg_pre_snap_entry_price", "avg_edge_to_90_cents", "latency_suspicion_score"]], formatters=formatters, max_rows=50)))


## Next Validation Query Shape

The next notebook should move from address-level ranking to fill-level event validation:

```text
address x event_slug x market_slug x winning_token x fill_timestamp x first_90c_timestamp x external_event_timestamp
```

The strongest candidates are fills where the external event is already true, the market has not yet printed $0.90, and the trader is an aggressing buyer rather than a passive maker fill.